# Mixed-class clustering · minority crop spectra at SR resolution

Different track from `pipeline.ipynb`'s LoRA augmentation. Hypothesis: super-resolution lets us separate the components of LDD's mixed-class polygons (e.g. `A403/A419` = Durian/Mangosteen) into pure-component clusters, so we can mine those polygons for extra single-class minority pixels.

Steps:
1. Inspect the shapefile · catalogue every `LU_CODE` including mixed (slash-separated) codes that involve a minority class.
2. Load one quadrant's SR cache + rasterise labels at the SR (2.5 m) grid, with each unique pure-or-mixed `LU_CODE` getting its own integer ID.
3. Sample per-pixel feature vectors (band stats + NDVI/NDWI/EVI).
4. Cluster pure minority pixels (KMeans + GMM, UMAP + HDBSCAN). Confirm each minority class forms a distinguishable cluster at SR resolution.
5. Project mixed-polygon pixels into the same embedding. Check if they split into sub-clusters matching their LU_CODE components.

If step 5 works, the mixed polygons become a free source of extra minority pixels — assigned to the closest pure cluster on a per-pixel basis.

In [1]:
from __future__ import annotations
import sys, warnings, struct
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import rasterio
import rioxarray as rxr
import xarray as xr
import geopandas as gpd
from rasterio.features import rasterize

warnings.filterwarnings("ignore", category=UserWarning)
print("python:", sys.version.split()[0])

python: 3.12.13


In [2]:
# >>> EDIT HERE <<<
QUADRANT       = "SE"                # which AOI's SR cache to read
STRIDE         = 4                   # SR-grid stride (lower = denser; 4 = every 4th SR pixel)
MAX_PIX_PER_CLASS = 100_000          # cap per (pure or mixed) LU_CODE; tune for RAM
MINORITY_TAXONOMY = {                # the minority A-codes we care about
    "A403": "Durian",
    "A404": "Rambutan",
    "A405": "Coconut",
    "A407": "Mango",
    "A413": "Longan",
    "A416": "Jackfruit",
    "A419": "Mangosteen",
    "A420": "Langsat",
}

REPO_ROOT  = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT  = REPO_ROOT / "data"
CACHE_ROOT = DATA_ROOT / "_cache"
SHP_DIR    = DATA_ROOT / "landuse_ryg"
SR_DIR     = CACHE_ROOT / "s2_sr" / f"rayong_{QUADRANT.lower()}"
OUT_DIR    = DATA_ROOT / "_out" / "cluster_minority"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"quadrant   {QUADRANT}")
print(f"SR cache   {SR_DIR}")
print(f"shapefile  {SHP_DIR}")
print(f"minority   {list(MINORITY_TAXONOMY.values())}")

quadrant   SE
SR cache   D:\Github\rayong-tracker\data\_cache\s2_sr\rayong_se
shapefile  D:\Github\rayong-tracker\data\landuse_ryg
minority   ['Durian', 'Rambutan', 'Coconut', 'Mango', 'Longan', 'Jackfruit', 'Mangosteen', 'Langsat']


## 1 · Shapefile inspection · mixed-class catalogue

LDD encodes mixed plots as slash-separated codes (`A403/A419` = Durian/Mangosteen on the same parcel). Catalogue every code that contains at least one minority A-code so we know what mixed combinations are available.

In [3]:
shp_files = list(SHP_DIR.rglob("*.shp"))
if not shp_files:
    raise FileNotFoundError(f"no .shp under {SHP_DIR}")
LU_SHP = shp_files[0]
print("shapefile:", LU_SHP.name)

lu_all = gpd.read_file(LU_SHP)
print("rows:", len(lu_all))
print("columns:", list(lu_all.columns))
print("crs:", lu_all.crs)

shapefile: LU_RYG_2567.shp


C:\Users\wttwk\anaconda3\envs\rayong-tracker\Lib\site-packages\pyogrio\core.py:35: RuntimeWarning: Could not detect PROJ data files. Set PROJ_LIB environment variable to the correct path.
  _init_proj_data()


rows: 65136
columns: ['LU_ID_L1', 'LU_ID_L2', 'LU_ID_L3', 'LU_CODE', 'LU_DES_TH', 'LU_DES_EN', 'LUL1_CODE', 'LUL2_CODE', 'LU_DES', 'Shape_Area', 'Area_Sqm', 'Area_Rai', 'geometry']
crs: PROJCS["WGS_1984_UTM_Zone_47N",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",99],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["Meter",1],AXIS["Easting",EAST],AXIS["Northing",NORTH]]


In [4]:
MINORITY_CODES = set(MINORITY_TAXONOMY)

def _components(code: str) -> list[str]:
    if not isinstance(code, str): return []
    return [c.strip() for c in code.split("/") if c.strip()]

lu_all["_components"] = lu_all["LU_CODE"].apply(_components)
lu_all["_n_components"] = lu_all["_components"].apply(len)
lu_all["_has_minority"] = lu_all["_components"].apply(
    lambda comps: any(c in MINORITY_CODES for c in comps)
)
# Stricter: mixed polygons we trust only when EVERY component is a minority
# (e.g. A403/A419 yes, A403/A411 = Durian/Banana no -- Banana is not a target).
lu_all["_all_minority"] = lu_all["_components"].apply(
    lambda comps: len(comps) > 0 and all(c in MINORITY_CODES for c in comps)
)

pure_minority  = lu_all[(lu_all["_n_components"] == 1) & lu_all["_all_minority"]]
mixed_minority = lu_all[(lu_all["_n_components"] >  1) & lu_all["_all_minority"]]

print(f"\npure minority polygons: {len(pure_minority)}")
print(f"mixed minority polygons: {len(mixed_minority)}\n")

vc_pure  = pure_minority ["LU_CODE"].value_counts()
vc_mixed = mixed_minority["LU_CODE"].value_counts()

print("pure minority by LU_CODE:")
for code, n in vc_pure.items():
    print(f"  {code:<10s} {MINORITY_TAXONOMY.get(code, '?'):<14s} {n:>5d}")

print("\nmixed combinations containing a minority class (top 30):")
for code, n in vc_mixed.head(30).items():
    comps = _components(code)
    names = [MINORITY_TAXONOMY.get(c, c) for c in comps]
    print(f"  {code:<14s} {' + '.join(names):<40s} {n:>5d}")


pure minority polygons: 7712
mixed minority polygons: 1830

pure minority by LU_CODE:
  A403       Durian          5585
  A405       Coconut          508
  A416       Jackfruit        448
  A419       Mangosteen       418
  A407       Mango            390
  A404       Rambutan         225
  A413       Longan           104
  A420       Langsat           34

mixed combinations containing a minority class (top 30):
  A403/A419      Durian + Mangosteen                        664
  A403/A411      Durian + A411                              226
  A403/A416      Durian + Jackfruit                         137
  A204/A405      A204 + Coconut                              95
  A403/A404      Durian + Rambutan                           87
  A302/A419      A302 + Mangosteen                           75
  A317/A419      A317 + Mangosteen                           71
  A404/A419      Rambutan + Mangosteen                       62
  A419/A420      Mangosteen + Langsat                        55
  A317/

## 2 · Load SR stack from cache · rasterise pure + mixed labels

Each unique `LU_CODE` (pure or mixed) gets its own integer ID in the label raster, so downstream pixel sampling can distinguish e.g. pure A403 (Durian) from A403/A419 (Durian/Mangosteen).

In [ ]:
tifs = sorted(SR_DIR.glob("sr_*.tif"))
if not tifs:
    raise FileNotFoundError(f"no SR tifs under {SR_DIR}")

sr_list = []
for p in tifs:
    ym = "".join(c for c in p.stem if c.isdigit())[:6]
    if len(ym) < 6: continue
    t = pd.to_datetime(ym, format="%Y%m")
    sr = rxr.open_rasterio(p, masked=True, chunks={"x": 2048, "y": 2048})
    sr_list.append(sr.expand_dims(time=[t]))
SR = xr.concat(sr_list, dim="time", join="exact")
print(f"SR stack {SR.shape} · res {SR.rio.resolution()} · crs {SR.rio.crs}")
print(f"months: {[str(t.values)[:7] for t in SR.time]}")

In [ ]:
lu = lu_all.to_crs(SR.rio.crs).copy()

# Unique-int per LU_CODE (pure or mixed). 0 = no label.
codes_in_use = sorted(lu[lu["_all_minority"]]["LU_CODE"].dropna().unique())
code_to_int = {code: i + 1 for i, code in enumerate(codes_in_use)}
int_to_code = {v: k for k, v in code_to_int.items()}
lu["_int"] = lu["LU_CODE"].map(code_to_int).fillna(0).astype("int32")

out_shape = SR.isel(time=0).shape[1:]
transform = SR.rio.transform()
shapes = [(g, v) for g, v in zip(lu.geometry, lu["_int"]) if v > 0]
LABELS = rasterize(shapes, out_shape=out_shape, transform=transform, fill=0, dtype="int32")
print(f"labels {LABELS.shape} · non-zero {(LABELS > 0).mean():.2%} · {len(codes_in_use)} distinct LU_CODEs in the AOI")

lbl_counts = pd.Series(LABELS[LABELS > 0]).value_counts()
print("\ntop 20 LU_CODEs by SR pixel count in this AOI:")
for i, n in lbl_counts.head(20).items():
    code = int_to_code[int(i)]
    comps = _components(code)
    names = [MINORITY_TAXONOMY.get(c, c) for c in comps]
    tag = "MIXED" if len(comps) > 1 else "pure "
    print(f"  {tag} {code:<14s} {' + '.join(names):<35s} {int(n):>8,}")

## 3 · Per-pixel features · band stats + NDVI / NDWI / EVI across the 3-month window

Same 12-dimensional schema as `pipeline.ipynb` so this notebook stays compatible with the RF feature space.

In [ ]:
rng = np.random.RandomState(0)
H, W = LABELS.shape
yy, xx = np.meshgrid(np.arange(0, H, STRIDE), np.arange(0, W, STRIDE), indexing="ij")
yy, xx = yy.ravel(), xx.ravel()
lbl = LABELS[yy, xx]
keep = lbl > 0
yy, xx, lbl = yy[keep], xx[keep], lbl[keep]

# Per-class cap.
keep_mask = np.zeros(len(yy), dtype=bool)
for cls_int in np.unique(lbl):
    idx = np.where(lbl == cls_int)[0]
    if len(idx) > MAX_PIX_PER_CLASS:
        idx = rng.choice(idx, size=MAX_PIX_PER_CLASS, replace=False)
    keep_mask[idx] = True
yy, xx, lbl = yy[keep_mask], xx[keep_mask], lbl[keep_mask]
print(f"sampled {len(yy):,} pixels (stride={STRIDE}, cap/class={MAX_PIX_PER_CLASS:,})")

n_t = len(SR.time)
spec = np.zeros((len(yy), n_t, 4), dtype="float32")
for ti in tqdm(range(n_t), desc="reading SR per month"):
    sr_t = SR.isel(time=ti).values
    if sr_t.max() > 5:
        sr_t = sr_t / 10000.0
    spec[:, ti, :] = sr_t[:, yy, xx].T
    del sr_t

_BAND_NAMES = ("B02", "B03", "B04", "B08")
stats: dict = {}
for b, name in enumerate(_BAND_NAMES):
    stats[f"{name}_mean"] = spec[:, :, b].mean(axis=1)
    stats[f"{name}_std"]  = spec[:, :, b].std(axis=1)

b02 = spec[:, :, 0]; b03 = spec[:, :, 1]; b04 = spec[:, :, 2]; b08 = spec[:, :, 3]
eps = 1e-6
ndvi = (b08 - b04) / (b08 + b04 + eps)
ndwi = (b03 - b08) / (b03 + b08 + eps)
evi  = 2.5 * (b08 - b04) / (b08 + 6.0 * b04 - 7.5 * b02 + 1.0 + eps)
stats["NDVI_mean"] = ndvi.mean(axis=1)
stats["NDVI_amp"]  = ndvi.max(axis=1) - ndvi.min(axis=1)
stats["NDWI_mean"] = ndwi.mean(axis=1)
stats["EVI_mean"]  = evi.mean(axis=1)

PIX = pd.DataFrame(stats)
PIX["label_int"] = lbl
PIX["lu_code"]   = [int_to_code[int(i)] for i in lbl]
PIX["is_mixed"]  = PIX["lu_code"].apply(lambda c: "/" in c)
PIX["primary_minority"] = PIX["lu_code"].apply(
    lambda c: next((MINORITY_TAXONOMY[x] for x in _components(c) if x in MINORITY_TAXONOMY), "?")
)
print(f"pixel table: {PIX.shape}")
PIX.head()

## 4 · Cluster pure minority pixels

Fit KMeans on pixels that come from pure-minority polygons only. One cluster per minority class as the target. Check if SR-grade features separate the classes cleanly — if not, the mixed-class decomposition step below won't work either.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

FEATURE_COLS = [c for c in PIX.columns if c not in ("label_int", "lu_code", "is_mixed", "primary_minority")]

pure = PIX[~PIX["is_mixed"]].copy()
print(f"pure minority pixels: {len(pure):,}")
print(pure["primary_minority"].value_counts().to_string())

scaler = StandardScaler().fit(pure[FEATURE_COLS].values)
X_pure = scaler.transform(pure[FEATURE_COLS].values)
K = pure["primary_minority"].nunique()
km = KMeans(n_clusters=K, random_state=0, n_init=10).fit(X_pure)
pure["cluster"] = km.labels_

ari = adjusted_rand_score(pure["primary_minority"], pure["cluster"])
nmi = normalized_mutual_info_score(pure["primary_minority"], pure["cluster"])
print(f"\nKMeans K={K} · ARI={ari:.3f} · NMI={nmi:.3f}")
print("\ncluster x class contingency:")
print(pd.crosstab(pure["cluster"], pure["primary_minority"]).to_string())

## 5 · 2-D embedding · UMAP scatter

Visualise the 12-D feature space in 2-D. Colour points by their true minority class, then by KMeans cluster. If the class-coloured plot shows separable blobs, the feature space carries enough signal for the mixed-class decomposition to work.

In [ ]:
try:
    import umap
    HAVE_UMAP = True
except Exception:
    HAVE_UMAP = False
    print("umap-learn not installed. `pip install umap-learn` for the 2-D plot.")

if HAVE_UMAP:
    SUBSAMPLE = min(20_000, len(pure))
    idx = np.random.RandomState(0).choice(len(pure), size=SUBSAMPLE, replace=False)
    X_sub = X_pure[idx]
    cls_sub = pure["primary_minority"].values[idx]
    clu_sub = pure["cluster"].values[idx]

    reducer = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=0)
    emb = reducer.fit_transform(X_sub)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, color_by, title in [
        (axes[0], cls_sub, "by minority class"),
        (axes[1], clu_sub, "by KMeans cluster"),
    ]:
        for v in sorted(np.unique(color_by)):
            m = color_by == v
            ax.scatter(emb[m, 0], emb[m, 1], s=3, alpha=0.5, label=str(v))
        ax.set_title(title)
        ax.legend(loc="upper right", fontsize=7, markerscale=2)
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(); plt.show()

## 6 · Mixed-polygon decomposition

For every mixed `LU_CODE` (e.g. `A403/A419` = Durian/Mangosteen), score each pixel's similarity to each pure-class centroid (from §4 KMeans). If SR resolution actually separates the mixed plot, the pixels should split into two sub-groups, one looking like Durian and the other looking like Mangosteen.

In [ ]:
# Per-class centroids (mean feature vector of each minority class' pure pixels).
centroids = {}
for cls_name in pure["primary_minority"].unique():
    cls_X = X_pure[pure["primary_minority"].values == cls_name]
    centroids[cls_name] = cls_X.mean(axis=0)

mixed = PIX[PIX["is_mixed"]].copy()
X_mixed = scaler.transform(mixed[FEATURE_COLS].values)

# Distance to every minority centroid.
names = list(centroids.keys())
C = np.stack([centroids[n] for n in names], axis=0)              # (K, D)
dist = np.linalg.norm(X_mixed[:, None, :] - C[None, :, :], axis=2)  # (N, K)
nearest_idx = dist.argmin(axis=1)
mixed["nearest_pure"] = [names[i] for i in nearest_idx]
mixed["nearest_dist"] = dist.min(axis=1)

print("For each mixed LU_CODE, how its pixels distribute across pure-class centroids:\n")
for code, sub in mixed.groupby("lu_code"):
    comps = _components(code)
    comp_names = [MINORITY_TAXONOMY.get(c, c) for c in comps]
    counts = sub["nearest_pure"].value_counts(normalize=True)
    if len(sub) < 50:
        continue
    top_share = counts.iloc[0]
    second_share = counts.iloc[1] if len(counts) > 1 else 0.0
    print(f"  {code:<14s} ({' + '.join(comp_names)})  N={len(sub):>5d}")
    for k, v in counts.head(3).items():
        marker = "✓" if k in comp_names else " "
        print(f"      {marker} -> {k:<14s} {v:>5.1%}")
    print(f"      top/second = {top_share:.2f} / {second_share:.2f}  (closer to 0.5/0.5 = clean two-way split)")

### 6.5 · Targeted mixed-pair clustering · `A_x / A_y` + pure `A_x` + pure `A_y`

Hypothesis: super-resolution lets pure-A and pure-B pixels separate, and mixed-A/B pixels split into two sub-groups one matching A and one matching B. Run K=2 KMeans on the union of three pixel groups (pure A, pure B, mixed A/B) and check (a) whether the K=2 boundary matches the pure labels, and (b) how the mixed pixels distribute across the two clusters.

If the K=2 cluster purity on pure-A and pure-B is high AND the mixed pixels split roughly 50/50, that is direct evidence the mixed plot is genuinely two-component and SR is separating them.


In [ ]:
# Targeted mixed-pair experiment.
TARGET_MIXED = "A403/A419"        # edit: A403/A419 = Durian/Mangosteen, A403/A416 Durian/Jackfruit, etc.
PURE_SAMPLE  = 5_000              # pixels sampled per pure component (cap)

_parts = [c.strip() for c in TARGET_MIXED.split("/") if c.strip()]
if len(_parts) != 2:
    raise ValueError(f"TARGET_MIXED must be exactly two slash-separated codes, got {TARGET_MIXED!r}")
_pa, _pb = _parts
if _pa not in MINORITY_TAXONOMY or _pb not in MINORITY_TAXONOMY:
    raise ValueError(f"Both components must be in MINORITY_TAXONOMY. Got {_pa}, {_pb}.")
_name_a = MINORITY_TAXONOMY[_pa]
_name_b = MINORITY_TAXONOMY[_pb]
print(f"target mixed code: {TARGET_MIXED}  ({_name_a} + {_name_b})")

_pure_a  = PIX[(~PIX["is_mixed"]) & (PIX["lu_code"] == _pa)]
_pure_b  = PIX[(~PIX["is_mixed"]) & (PIX["lu_code"] == _pb)]
_mixed_ab = PIX[PIX["lu_code"] == TARGET_MIXED]
print(f"  pure {_pa} {_name_a:<12s} pixels in AOI: {len(_pure_a):,}")
print(f"  pure {_pb} {_name_b:<12s} pixels in AOI: {len(_pure_b):,}")
print(f"  mixed {TARGET_MIXED:<10s} pixels in AOI: {len(_mixed_ab):,}")

if len(_pure_a) == 0 or len(_pure_b) == 0 or len(_mixed_ab) == 0:
    raise RuntimeError(
        f"Need pure pixels for both {_pa} and {_pb} AND mixed pixels for {TARGET_MIXED} in this AOI. "
        f"Either change TARGET_MIXED or rerun on a different QUADRANT."
    )

_rng_p = np.random.RandomState(7)
def _take(df: pd.DataFrame, n: int) -> pd.DataFrame:
    if len(df) <= n: return df
    idx = _rng_p.choice(len(df), size=n, replace=False)
    return df.iloc[idx]

_pure_a_s  = _take(_pure_a,  PURE_SAMPLE)
_pure_b_s  = _take(_pure_b,  PURE_SAMPLE)
_mixed_s   = _take(_mixed_ab, max(PURE_SAMPLE, 2_000))

_pool = pd.concat([
    _pure_a_s.assign(_origin=f"pure {_name_a}"),
    _pure_b_s.assign(_origin=f"pure {_name_b}"),
    _mixed_s.assign(_origin=f"mixed {TARGET_MIXED}"),
], ignore_index=True)

_X_pool = scaler.transform(_pool[FEATURE_COLS].values)

# K=2 on the pooled pixels.
from sklearn.cluster import KMeans as _KMeans
_km2 = _KMeans(n_clusters=2, random_state=0, n_init=10).fit(_X_pool)
_pool["cluster"] = _km2.labels_

# Align cluster IDs with the pure components by majority vote.
_a_mask = _pool["_origin"].str.startswith(f"pure {_name_a}")
_b_mask = _pool["_origin"].str.startswith(f"pure {_name_b}")
_cluster_for_a = int(_pool.loc[_a_mask, "cluster"].mode().iloc[0])
_cluster_for_b = 1 - _cluster_for_a
_pool["cluster_name"] = _pool["cluster"].map({_cluster_for_a: _name_a, _cluster_for_b: _name_b})

print("\nK=2 cluster purity on pure components:")
for nm, mask in [(_name_a, _a_mask), (_name_b, _b_mask)]:
    sub = _pool[mask]
    correct = (sub["cluster_name"] == nm).mean()
    print(f"  pure {nm:<12s} N={len(sub):,}  -> assigned to {nm:<12s}: {correct:.1%}")

print("\nMixed-polygon pixel split across the K=2 boundary:")
mixed_pool = _pool[_pool["_origin"].str.startswith("mixed")]
share = mixed_pool["cluster_name"].value_counts(normalize=True)
for k, v in share.items():
    print(f"  {k:<12s} {v:.1%}")
print(f"  N = {len(mixed_pool):,}")

# UMAP scatter (3 colours by origin, then K=2 by cluster).
if HAVE_UMAP:
    _emb = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=0).fit_transform(_X_pool)
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    palette = {
        f"pure {_name_a}": "#3F7D58",
        f"pure {_name_b}": "#C96442",
        f"mixed {TARGET_MIXED}": "#7B5BA6",
    }
    for v, mk in palette.items():
        m = _pool["_origin"] == v
        axes[0].scatter(_emb[m, 0], _emb[m, 1], s=4, alpha=0.5, label=v, c=mk)
    axes[0].set_title(f"by origin · {TARGET_MIXED}")
    axes[0].legend(loc="upper right", fontsize=8, markerscale=2)

    cluster_palette = {_name_a: "#3F7D58", _name_b: "#C96442"}
    for v, mk in cluster_palette.items():
        m = _pool["cluster_name"] == v
        axes[1].scatter(_emb[m, 0], _emb[m, 1], s=4, alpha=0.5, label=f"cluster -> {v}", c=mk)
    axes[1].set_title(f"K=2 cluster assignment")
    axes[1].legend(loc="upper right", fontsize=8, markerscale=2)
    for ax in axes:
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(); plt.show()


## 7 · Decision · use mixed-polygon pixels as extra minority data?

Read the §6 table. Three regimes:
- **Clean split** (e.g. `A403/A419` shows ~50 % Durian / ~50 % Mangosteen with the ✓ marker on both) → SR separates the components, append those pixels to RF training with the per-pixel nearest-centroid label.
- **One-sided** (one component takes >85 % of pixels) → the mixed polygon is dominated by one class on this AOI / time window. Use only that class as the label for all of its pixels.
- **Garbage** (top class is not in the polygon's component list) → SR isn't separating these classes well in feature space; skip the polygon.

Optional next step: persist the labelled mixed-polygon pixels to `data/_out/cluster_minority/<aoi>_mixed_pixels.parquet` and append to the RF feature table in `pipeline.ipynb` (§8 reload path).

In [ ]:
# Tag every mixed pixel with the inferred per-pixel label (nearest pure centroid),
# but only when that nearest centroid is one of the polygon's listed components.
def _inferred_label(row) -> str | None:
    comps = _components(row["lu_code"])
    comp_names = [MINORITY_TAXONOMY.get(c, c) for c in comps]
    return row["nearest_pure"] if row["nearest_pure"] in comp_names else None

mixed["inferred_class"] = mixed.apply(_inferred_label, axis=1)
kept = mixed[mixed["inferred_class"].notna()].copy()
print(f"mixed pixels with valid inferred class: {len(kept):,} / {len(mixed):,}")
print("\ninferred class breakdown:")
print(kept["inferred_class"].value_counts().to_string())

out_cols = FEATURE_COLS + ["label_int", "lu_code", "nearest_pure", "nearest_dist", "inferred_class"]
out_path = OUT_DIR / f"rayong_{QUADRANT.lower()}_mixed_pixels.parquet"
try:
    kept[out_cols].to_parquet(out_path)
    print(f"\nwrote {out_path}")
except Exception as e:
    out_path = out_path.with_suffix(".pkl")
    kept[out_cols].to_pickle(out_path)
    print(f"\nparquet unavailable ({type(e).__name__}); wrote pickle: {out_path}")